In [ ]:
import json
import pandas as pd

with open("../../dataset/train_sentiment.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../../"))

In [ ]:
from utility.preprocess import clean_text
text = df['text']
sentiment = df['sentiment']

# text = df["text"].apply(clean_text)

In [ ]:
from sklearn.model_selection import train_test_split

text_train, text_test, sentiment_train, sentiment_test = train_test_split(
    text, sentiment, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import StackingClassifier

base_models = [
    ("svm", LinearSVC()),
    ("nb", MultinomialNB()),
    # ("lr", LogisticRegression(max_iter=1000)),
    ("sgd", SGDClassifier(loss="log_loss"))
]

meta_model = LogisticRegression()

In [ ]:
stack_model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from utility.preprocess import tokenizer

vectorizer = TfidfVectorizer(
    tokenizer=tokenizer,
    ngram_range=(1,2),
    # min_df=3,
    # max_df=0.9
)

text_train_tfidf = vectorizer.fit_transform(text_train)
text_val_tfidf = vectorizer.transform(text_test)

In [ ]:
import numpy as np
from utility.preprocess import lexicon_features

lex_features_train = np.array([lexicon_features(t) for t in text_train])
lex_features_test = np.array([lexicon_features(t) for t in text_test])

In [ ]:
from scipy.sparse import hstack

train_combined = hstack([text_train_tfidf, lex_features_train])
test_combined = hstack([text_val_tfidf, lex_features_test])

In [ ]:
stack_model.fit(train_combined, sentiment_train)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report # type: ignore

y_pred = stack_model.predict(test_combined)

print("Accuracy:", accuracy_score(sentiment_test, y_pred))
print(classification_report(sentiment_test, y_pred))